In [ ]:
import polars as pl
from pathlib import Path

In [ ]:
info_path = Path("/home/ubuntu/patent_similarity_new/data/stkcd_info.xlsx")
check_cols = ["province", "city", "Ind"]

target_stkcd = None


def clean_text_col(name: str) -> pl.Expr:
    cleaned = pl.col(name).cast(pl.Utf8).str.strip_chars()
    return (
        pl.when(pl.col(name).is_null() | (cleaned == ""))
        .then(pl.lit(None, dtype=pl.Utf8))
        .otherwise(cleaned)
        .alias(name)
    )


stkcd_info = (
    pl.read_excel(info_path)
    .with_columns(
        pl.col("stkcd").cast(pl.Utf8).str.zfill(6),
        pl.col("year").cast(pl.Int64, strict=False),
        *[clean_text_col(c) for c in check_cols],
    )
)

scan_base = (
    stkcd_info
    if target_stkcd is None
    else stkcd_info.filter(pl.col("stkcd") == str(target_stkcd).zfill(6))
)

stkcd_changes = (
    scan_base.group_by("stkcd")
    .agg(
        pl.col("year").min().alias("first_year"),
        pl.col("year").max().alias("last_year"),
        pl.len().alias("n_rows"),
        *[pl.col(c).drop_nulls().n_unique().alias(f"n_{c}") for c in check_cols],
        *[
            pl.col(c).drop_nulls().unique().sort().alias(f"{c}_values")
            for c in check_cols
        ],
    )
    .with_columns(
        pl.any_horizontal([pl.col(f"n_{c}") > 1 for c in check_cols]).alias(
            "has_different_province_city_ind"
        )
    )
    .filter(pl.col("has_different_province_city_ind"))
    .sort("stkcd")
)

stkcd_change_details = (
    stkcd_info.join(stkcd_changes.select("stkcd"), on="stkcd", how="semi")
    .select("stkcd", "year", *check_cols)
    .unique()
    .sort(["stkcd", "year"])
)